# Pocket Barrister: first provisional Gemma 2B QLoRA run

This notebook trains and evaluates the first canonical-pipeline adapter. The dataset is synthetic and legally unreviewed. The run measures formatting and inherited-state reproduction; it does **not** establish legal correctness or provide legal advice.

Before running: select a GPU runtime, accept the Gemma terms on Hugging Face, add `HF_TOKEN` to Colab secrets, and replace `REPO_URL` below after pushing the repository.

In [ ]:
# 1. Clone the exact repository revision you want to train.
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/YOUR_USERNAME/PocketBarrister.git"
REPO_REVISION = "main"  # Prefer a commit SHA after your next push.
REPO_DIR = Path("/content/PocketBarrister")
assert "YOUR_USERNAME" not in REPO_URL, "Set REPO_URL before continuing."
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "fetch", "origin", REPO_REVISION], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
REPO_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Repository commit:", REPO_COMMIT)

In [ ]:
# 2. Install a pinned, conservative QLoRA stack. The Colab torch/CUDA build is recorded later.
subprocess.run([
    "python", "-m", "pip", "install", "-q",
    "transformers==4.51.3",
    "peft==0.15.2",
    "accelerate==1.6.0",
    "bitsandbytes==0.45.5",
    "huggingface_hub==0.30.2",
    "sentencepiece==0.2.0",
    "PyYAML==6.0.2",
], check=True)

In [ ]:
# 3. Rebuild and verify data before allocating GPU memory.
for command in (["python", "-B", "analysis/verify_legacy_snapshot.py"],
                ["python", "-B", "scripts/build_dataset.py"],
                ["python", "-B", "scripts/validate_dataset.py"]):
    subprocess.run(command, check=True)

In [ ]:
# 4. Authenticate, resolve the mutable model name to an immutable commit, and seed everything.
import json, random, sys, yaml, torch
from google.colab import userdata
from huggingface_hub import HfApi, login

sys.path.insert(0, str(REPO_DIR / "src"))
from pocket_barrister.training.formatting import format_prompt, format_target

HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Add a Hugging Face read token as the HF_TOKEN Colab secret."
login(token=HF_TOKEN, add_to_git_credential=False)
config = yaml.safe_load(Path("configs/gemma_2b_qlora_provisional_v0.yaml").read_text())
manifest = json.loads(Path("data/provisional_v0/manifest.json").read_text())
BASE_MODEL = config["base_model"]
BASE_REVISION = HfApi().model_info(BASE_MODEL, revision="main", token=HF_TOKEN).sha
SEED = int(config["training"]["seed"])
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
assert torch.cuda.is_available(), "Select a GPU runtime."
print({"base_model": BASE_MODEL, "base_revision": BASE_REVISION, "gpu": torch.cuda.get_device_name(0)})

In [ ]:
# 5. Load family-disjoint records and define response-only tokenization.
from torch.utils.data import Dataset
from transformers import AutoTokenizer

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line]

train_rows = read_jsonl("data/provisional_v0/splits/train.jsonl")
validation_rows = read_jsonl("data/provisional_v0/splits/validation.jsonl")
test_rows = read_jsonl("data/provisional_v0/splits/test.jsonl")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_REVISION, token=HF_TOKEN)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
MAX_LENGTH = int(config["training"]["max_length"])

class ResponseOnlyDataset(Dataset):
    def __init__(self, rows):
        self.items = [self.encode(row) for row in rows]
    def encode(self, row):
        prompt = format_prompt(row["input"])
        target = format_target(row["output"], tokenizer.eos_token)
        prompt_ids = tokenizer(prompt, add_special_tokens=True)["input_ids"]
        encoded = tokenizer(prompt + target, add_special_tokens=True, truncation=True, max_length=MAX_LENGTH)
        labels = encoded["input_ids"].copy()
        labels[:min(len(prompt_ids), len(labels))] = [-100] * min(len(prompt_ids), len(labels))
        encoded["labels"] = labels
        return encoded
    def __len__(self): return len(self.items)
    def __getitem__(self, index): return self.items[index]

train_dataset = ResponseOnlyDataset(train_rows)
validation_dataset = ResponseOnlyDataset(validation_rows)
assert all(any(label != -100 for label in item["labels"]) for item in train_dataset.items)
print({"train": len(train_rows), "validation": len(validation_rows), "test": len(test_rows)})

In [ ]:
# 6. Load Gemma in 4-bit and attach a fresh LoRA adapter.
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, revision=BASE_REVISION, token=HF_TOKEN,
    quantization_config=quantization, device_map="auto",
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora = config["lora"]
model = get_peft_model(model, LoraConfig(
    r=int(lora["rank"]), lora_alpha=int(lora["alpha"]),
    lora_dropout=float(lora["dropout"]), bias=lora["bias"],
    target_modules=lora["target_modules"], task_type="CAUSAL_LM",
))
model.config.use_cache = False
model.print_trainable_parameters()

In [ ]:
# 7. Freeze prompted-base predictions before optimization.
@torch.inference_mode()
def generate_rows(rows, system):
    predictions = []
    model.eval()
    for row in rows:
        encoded = tokenizer(format_prompt(row["input"]), return_tensors="pt").to(model.device)
        generated = model.generate(
            **encoded, do_sample=False, max_new_tokens=int(config["evaluation"]["max_new_tokens"]),
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
        )[0, encoded["input_ids"].shape[1]:]
        predictions.append({
            "sample_id": row["sample_id"], "system": system,
            "prediction": tokenizer.decode(generated, skip_special_tokens=True).strip(),
            "ended_with_eos": bool(len(generated) and generated[-1].item() == tokenizer.eos_token_id),
            "generated_token_count": int(len(generated)),
        })
    return predictions

with model.disable_adapter():
    base_predictions = generate_rows(test_rows, "base")
print("Stored prompted-base predictions:", len(base_predictions))

In [ ]:
# 8. Train with prompt tokens masked from loss and select the lowest validation loss.
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments

training = config["training"]
args = TrainingArguments(
    output_dir="outputs/pb-gemma-2b-qlora-provisional-v0",
    num_train_epochs=float(training["epochs"]),
    learning_rate=float(training["learning_rate"]),
    per_device_train_batch_size=int(training["per_device_train_batch_size"]),
    per_device_eval_batch_size=int(training["per_device_eval_batch_size"]),
    gradient_accumulation_steps=int(training["gradient_accumulation_steps"]),
    warmup_ratio=float(training["warmup_ratio"]), weight_decay=float(training["weight_decay"]),
    optim=training["optimizer"], eval_strategy="epoch", save_strategy="epoch",
    logging_steps=1, load_best_model_at_end=True, metric_for_best_model="eval_loss",
    greater_is_better=False, report_to="none", seed=SEED, data_seed=SEED,
    fp16=compute_dtype == torch.float16, bf16=compute_dtype == torch.bfloat16,
    gradient_checkpointing=True, save_total_limit=2,
)
collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model, padding=True, pad_to_multiple_of=8, label_pad_token_id=-100
)
trainer = Trainer(
    model=model, args=args, train_dataset=train_dataset, eval_dataset=validation_dataset,
    data_collator=collator, processing_class=tokenizer,
)
train_result = trainer.train()
train_result.metrics

In [ ]:
# 9. Save adapter, immutable run metadata, environment, and raw adapter predictions.
import platform
from datetime import datetime, timezone

adapter_dir = Path(config["artifacts"]["adapter_dir"])
results_dir = Path(config["artifacts"]["predictions_dir"])
adapter_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(adapter_dir, safe_serialization=True)
tokenizer.save_pretrained(adapter_dir)
adapter_predictions = generate_rows(test_rows, "adapter")
prediction_path = results_dir / "predictions.jsonl"
with prediction_path.open("w", encoding="utf-8") as handle:
    for row in base_predictions + adapter_predictions:
        handle.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + "\n")
run_metadata = {
    "experiment_id": config["experiment_id"], "created_at": datetime.now(timezone.utc).isoformat(),
    "research_status": config["research_status"], "repository_commit": REPO_COMMIT,
    "base_model": BASE_MODEL, "base_revision": BASE_REVISION,
    "dataset_sha256": manifest["dataset_sha256"], "split_sha256": manifest["split_sha256"],
    "seed": SEED, "gpu": torch.cuda.get_device_name(0), "python": platform.python_version(),
    "torch": torch.__version__, "cuda": torch.version.cuda, "train_metrics": train_result.metrics,
}
(results_dir / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2, sort_keys=True) + "\n")
(results_dir / "environment.txt").write_text(subprocess.check_output(["python", "-m", "pip", "freeze"], text=True))
print(run_metadata)

In [ ]:
# 10. Score both systems with the same transparent structural metrics.
details_path = results_dir / "scores.jsonl"
score_output = subprocess.check_output([
    "python", "-B", "scripts/score_predictions.py", str(prediction_path),
    "--details", str(details_path),
], text=True)
(results_dir / "summary_metrics.json").write_text(score_output, encoding="utf-8")
print(score_output)

In [ ]:
# 11. Bundle the adapter and evidence for download. Inspect before publishing anything.
import shutil
from google.colab import files

bundle_root = Path("/content/pocket_barrister_run")
if bundle_root.exists(): shutil.rmtree(bundle_root)
bundle_root.mkdir()
shutil.copytree(adapter_dir, bundle_root / "adapter")
shutil.copytree(results_dir, bundle_root / "results")
archive = shutil.make_archive("/content/pocket_barrister_run", "zip", bundle_root)
files.download(archive)